In [4]:
# Imports and configuration

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold
)

from sklearn.preprocessing import (
    OneHotEncoder,
    MultiLabelBinarizer
)

from scipy.sparse import (
    hstack,
    csr_matrix
)

RANDOM_STATE = 42

DATASET_PATH = "../data/processed/nliceCleaned.csv"

print("Configuration loaded.")
print("Random state:", RANDOM_STATE)

Configuration loaded.
Random state: 42


In [5]:
# Load cleaned NLICE dataset

cleaned = pd.read_csv(
    DATASET_PATH,
    dtype={
        "PATIENT": "string",
        "AGE": "int16",
        "GENDER": "string",
        "SYMPTOMS_CLEAN": "string",
        "LOCATION": "string",
        "PATHOLOGY": "string"
    }
)

print("Dataset shape:", cleaned.shape)
display(cleaned.head())

Dataset shape: (3692607, 6)


,PATIENT,AGE,GENDER,SYMPTOMS_CLEAN,LOCATION,PATHOLOGY
0,b0915fdc-11f2-430a-aece-4f19f2d03b0f,40,M,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
1,bc8b5a00-f5cd-4675-bd26-68ff53101121,58,F,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
2,611a810d-eed5-4156-9d98-e375b3754df0,22,M,Inflammation|Congestion|Discharge|Altered_sens...,Head:Ear|Head:Ear|Head:Ear|Head:Ear,Otitis externa
3,41542e6c-0b9c-464e-bb74-00f52b511f07,59,M,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa
4,280423b0-3df5-416e-b446-0208679936bd,18,F,Congestion|Discharge|Altered_sensation,Head:Ear|Head:Ear|Head:Ear,Otitis externa


In [6]:
# Basic dataset validation

print("Rows:", len(cleaned))
print("Columns:", cleaned.columns.tolist())

print("\nMissing values:")
print(cleaned.isna().sum())

print("\nUnique patients:", cleaned["PATIENT"].nunique())
print("Unique diseases:", cleaned["PATHOLOGY"].nunique())
print("Unique genders:", cleaned["GENDER"].nunique())
print("Unique symptoms:", cleaned["SYMPTOMS_CLEAN"].nunique())
print("Unique locations:", cleaned["LOCATION"].nunique())

print("\nAge range:")
print(cleaned["AGE"].min(), "to", cleaned["AGE"].max())

Rows: 3692607
Columns: ['PATIENT', 'AGE', 'GENDER', 'SYMPTOMS_CLEAN', 'LOCATION', 'PATHOLOGY']

Missing values:
PATIENT                0
AGE                    0
GENDER                 0
SYMPTOMS_CLEAN         0
LOCATION          457066
PATHOLOGY              0
dtype: int64

Unique patients: 550363
Unique diseases: 55
Unique genders: 2
Unique symptoms: 14555
Unique locations: 5336

Age range:
0 to 109


In [7]:
# Patient-Level 80/20 Train-Test Split
#
# The test set is locked here and will NOT be used
# during cross-validation or hyperparameter tuning.

X = cleaned.drop(columns=["PATHOLOGY"])
y = cleaned["PATHOLOGY"]
groups = cleaned["PATIENT"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

train_data = cleaned.iloc[train_idx].copy()
test_data = cleaned.iloc[test_idx].copy()

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

print("Train patients:", train_data["PATIENT"].nunique())
print("Test patients:", test_data["PATIENT"].nunique())

Train shape: (2952952, 6)
Test shape: (739655, 6)
Train patients: 440290
Test patients: 110073


In [8]:
# Verify patient-level holdout integrity

train_patients = set(train_data["PATIENT"])
test_patients = set(test_data["PATIENT"])

patient_overlap = train_patients.intersection(test_patients)

print("Patient overlap:", len(patient_overlap))

print("\nTrain diseases:", train_data["PATHOLOGY"].nunique())
print("Test diseases:", test_data["PATHOLOGY"].nunique())

missing_in_test = set(train_data["PATHOLOGY"]) - set(test_data["PATHOLOGY"])
missing_in_train = set(test_data["PATHOLOGY"]) - set(train_data["PATHOLOGY"])

print("\nDiseases missing from test:", len(missing_in_test))
print("Diseases missing from train:", len(missing_in_train))

assert len(patient_overlap) == 0
assert len(missing_in_test) == 0
assert len(missing_in_train) == 0

print("\nHoldout validation passed.")

Patient overlap: 0

Train diseases: 55
Test diseases: 55

Diseases missing from test: 0
Diseases missing from train: 0

Holdout validation passed.


In [9]:
# Prepare raw features and targets

X_train_raw = train_data.drop(
    columns=["PATIENT", "PATHOLOGY"]
).copy()

X_test_raw = test_data.drop(
    columns=["PATIENT", "PATHOLOGY"]
).copy()

y_train = train_data["PATHOLOGY"].copy()
y_test = test_data["PATHOLOGY"].copy()

groups_train = train_data["PATIENT"].copy()

print("Training features:", X_train_raw.shape)
print("Testing features:", X_test_raw.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (2952952, 4)
Testing features: (739655, 4)
Training target: (2952952,)
Testing target: (739655,)


In [10]:
# Encode Age and Gender

# -----------------------------
# Age
# -----------------------------

age_train = csr_matrix(
    X_train_raw[["AGE"]]
    .astype("float32")
    .values
)

age_test = csr_matrix(
    X_test_raw[["AGE"]]
    .astype("float32")
    .values
)

# -----------------------------
# Gender
# -----------------------------

gender_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

gender_train = gender_encoder.fit_transform(
    X_train_raw[["GENDER"]]
)

gender_test = gender_encoder.transform(
    X_test_raw[["GENDER"]]
)

print("Age features:", age_train.shape[1])
print("Gender features:", gender_train.shape[1])
print("Gender categories:", gender_encoder.categories_)

Age features: 1
Gender features: 2
Gender categories: [array(['F', 'M'], dtype=object)]


In [11]:
# Encode Symptoms and Locations

# -----------------------------
# Prepare missing values
# -----------------------------

X_train_raw["SYMPTOMS_CLEAN"] = (
    X_train_raw["SYMPTOMS_CLEAN"]
    .fillna("")
)

X_test_raw["SYMPTOMS_CLEAN"] = (
    X_test_raw["SYMPTOMS_CLEAN"]
    .fillna("")
)

X_train_raw["LOCATION"] = (
    X_train_raw["LOCATION"]
    .fillna("UNKNOWN")
)

X_test_raw["LOCATION"] = (
    X_test_raw["LOCATION"]
    .fillna("UNKNOWN")
)

# -----------------------------
# Symptoms
# -----------------------------

symptom_encoder = MultiLabelBinarizer(
    sparse_output=True
)

train_symptom_lists = (
    X_train_raw["SYMPTOMS_CLEAN"]
    .str.split("|")
)

test_symptom_lists = (
    X_test_raw["SYMPTOMS_CLEAN"]
    .str.split("|")
)

symptoms_train = symptom_encoder.fit_transform(
    train_symptom_lists
)

symptoms_test = symptom_encoder.transform(
    test_symptom_lists
)

# -----------------------------
# Locations
# -----------------------------

location_encoder = MultiLabelBinarizer(
    sparse_output=True
)

train_location_lists = (
    X_train_raw["LOCATION"]
    .str.split("|")
)

test_location_lists = (
    X_test_raw["LOCATION"]
    .str.split("|")
)

locations_train = location_encoder.fit_transform(
    train_location_lists
)

locations_test = location_encoder.transform(
    test_location_lists
)

print("Symptom features:", symptoms_train.shape[1])
print("Location features:", locations_train.shape[1])

Symptom features: 80
Location features: 137


In [12]:
# Combine all encoded features

X_train = hstack([
    age_train,
    gender_train,
    symptoms_train,
    locations_train
]).tocsr()

X_test = hstack([
    age_test,
    gender_test,
    symptoms_test,
    locations_test
]).tocsr()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (2952952, 220)
X_test shape: (739655, 220)


In [13]:
# Final feature representation checks

print("========== FEATURE SUMMARY ==========")

print("Training samples :", X_train.shape[0])
print("Testing samples  :", X_test.shape[0])
print("Total features   :", X_train.shape[1])

print("\nSparse matrix:")
print("X_train:", type(X_train))
print("X_test :", type(X_test))

print("\nNon-zero values:")
print("X_train:", X_train.nnz)
print("X_test :", X_test.nnz)

print("\nTargets:")
print("Training classes:", y_train.nunique())
print("Testing classes :", y_test.nunique())

assert X_train.shape[1] == X_test.shape[1]
assert y_train.nunique() == 55
assert y_test.nunique() == 55

print("\nFeature preparation successful.")

========== FEATURE SUMMARY ==========
Training samples : 2952952
Testing samples  : 739655
Total features   : 220

Sparse matrix:
X_train: <class 'scipy.sparse._csr.csr_matrix'>
X_test : <class 'scipy.sparse._csr.csr_matrix'>

Non-zero values:
X_train: 20543203
X_test : 5144241

Targets:
Training classes: 55
Testing classes : 55

Feature preparation successful.


In [14]:
# Grouped Cross-Validation Strategy
#
# Cross-validation is performed ONLY on the 80% training data.
# PATIENT is used as the grouping variable so that records
# from the same patient never appear in both train and validation
# folds.

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Cross-validation strategy:")
print(cv)

print("\nNumber of folds:", cv.n_splits)
print("Grouping variable: PATIENT")
print("Stratification target: PATHOLOGY")

Cross-validation strategy:
StratifiedGroupKFold(n_splits=3, random_state=42, shuffle=True)

Number of folds: 3
Grouping variable: PATIENT
Stratification target: PATHOLOGY


In [ ]:
# Processing complete

print("==========================================")
print("NLICE PROCESSING COMPLETE")
print("==========================================")

print(f"Original dataset : {len(cleaned):,} records")
print(f"Training records : {len(train_data):,}")
print(f"Testing records  : {len(test_data):,}")

print(f"\nTraining patients: {train_data['PATIENT'].nunique():,}")
print(f"Testing patients : {test_data['PATIENT'].nunique():,}")

print(f"\nDisease classes  : {y_train.nunique()}")
print(f"Total features   : {X_train.shape[1]}")

print("\nFeature composition:")
print("  AGE       →", age_train.shape[1])
print("  GENDER    →", gender_train.shape[1])
print("  SYMPTOMS  →", symptoms_train.shape[1])
print("  LOCATION  →", locations_train.shape[1])

print("\nValidation:")
print("  Patient overlap → 0")
print("  CV folds        → 3")
print("  CV grouping     → PATIENT")
print("  CV stratifying  → PATHOLOGY")

NLICE PROCESSING COMPLETE
Original dataset : 3,692,607 records
Training records : 2,952,952
Testing records  : 739,655

Training patients: 440,290
Testing patients : 110,073

Disease classes  : 55
Total features   : 220

Feature composition:
  AGE       → 1
  GENDER    → 2
  SYMPTOMS  → 80
  LOCATION  → 137

Validation:
  Patient overlap → 0
  CV folds        → 3
  CV grouping     → PATIENT
  CV stratifying  → PATHOLOGY

Ready for finalModel.ipynb.
